In [ ]:
pip install selenium pandas webdriver-manager

In [ ]:


import sys
import time
import random
import pandas as pd

sys.path.insert(0, '/usr/lib/chromium-browser/chromedriver')

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC



PRODUCT_URL = "https://www.daraz.com.bd/products/4-i343927190-s1677562400.html?pvid=ae0acae0-74ab-434d-a79e-bdbdd191e30f&search=jfy&scm=1007.51705.413671.0&spm=a2a0e.tm80335411.just4u.d_343927190"
MAX_PAGES = 3
OUTPUT_FILE = "daraz_reviews.csv"



options = webdriver.ChromeOptions()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-gpu')
options.add_argument('--window-size=1920,1080')

options.binary_location = '/usr/bin/chromium-browser'

service = Service('/usr/lib/chromium-browser/chromedriver')
driver = webdriver.Chrome(service=service, options=options)

wait = WebDriverWait(driver, 15)



driver.get(PRODUCT_URL)
time.sleep(5)



def slow_scroll():
    for _ in range(5):
        driver.execute_script("window.scrollBy(0, 800);")
        time.sleep(random.uniform(1, 2))



try:
    review_tab = wait.until(
        EC.element_to_be_clickable((By.XPATH, "//a[contains(text(),'Reviews')]"))
    )
    review_tab.click()
    time.sleep(3)
except:
    print("Review tab not found (may auto-load).")



all_reviews = []
current_page = 1

while current_page <= MAX_PAGES:
    print(f"Scraping page {current_page}...")
    slow_scroll()
    time.sleep(3)

   
    review_blocks = driver.find_elements(By.CLASS_NAME, "item")

    for block in review_blocks:
        try:
            review_text = block.find_element(By.CLASS_NAME, "content").text
        except:
            review_text = ""

        try:
            date = block.find_element(By.CLASS_NAME, "middle").text
        except:
            date = ""

        if review_text.strip():
            all_reviews.append({
                "review_text": review_text,
                "date": date,
                "review_length": len(review_text)
            })

    
    try:
        next_button = driver.find_element(By.XPATH, "//button[contains(@class,'next')]")
        driver.execute_script("arguments[0].click();", next_button)
        current_page += 1
        time.sleep(4)
    except:
        print("No more pages.")
        break



df = pd.DataFrame(all_reviews)
df.drop_duplicates(inplace=True)
df.to_csv(OUTPUT_FILE, index=False)

print("Total Reviews Saved:", len(df))
driver.quit()

In [ ]:
import requests
import pandas as pd
import time


PRODUCT_ID = "416533626"
MAX_PAGES = 5
OUTPUT_FILE = "daraz_reviews_416533626.csv"

all_reviews = []

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
}

for page in range(1, MAX_PAGES + 1):
    print(f"Fetching page {page}...")
    
    url = f"https://my.daraz.com.bd/pdp/review/getReviewList?itemId={PRODUCT_ID}&pageSize=5&pageNo={page}"
    
    response = requests.get(url, headers=headers)
    
    if response.status_code != 200:
        print("Request failed")
        break
    
    data = response.json()
    
    if "model" not in data or "items" not in data["model"]:
        print("No more reviews")
        break
    
    reviews = data["model"]["items"]
    
    for review in reviews:
        review_text_content = review.get("reviewContent", "")
    
        if review_text_content is None:
            review_text_content = ""

        all_reviews.append({
            "review_text": review_text_content,
            "rating": review.get("rating", ""),
            "date": review.get("reviewTime", ""),
            "review_length": len(review_text_content)
        })
    
    time.sleep(2)

df = pd.DataFrame(all_reviews)
df.drop_duplicates(inplace=True)
df.to_csv(OUTPUT_FILE, index=False)

print("Total Reviews Collected:", len(df))